# FeynKit one-loop quickstart

This ten-minute tour goes from a normalized quantum-field-theory model to a typed one-loop \(1 \to 2\) scalar amplitude. FeynKit is part of the same `symbolica` installation and shares Symbolica's expression engine.

## Setup

Install the community distribution with `pip install symbolica`. We use the conventional short alias `fk` so the physics namespace stays visible.

In [ ]:
from pathlib import Path

import symbolica.community.feynkit as fk

DATA = next(path for path in (Path("data"), Path("examples/feynkit/data")) if path.exists())
print(fk.__name__)

## Load a normalized model

A normalized JSON model is portable and does not require Python UFO tooling. The bundled scalar model is deliberately small enough for an interactive tutorial.

In [ ]:
model = fk.Model(DATA / "scalars_2p_3p.json")
{
    "model": model.name,
    "particles": len(model.particles),
    "parameters": len(model.parameters),
    "vertices": len(model.vertex_rules),
}

## Describe and generate an amplitude

Particle selectors may be names such as `"scalar_0"`, PDG codes, or explicit `ParticleSelector` objects. Here `loops=1` requests exactly one loop. Enabling self-loops lets the generator enumerate the complete set of allowed one-loop topologies; below, we select the first diagram without a self-edge for a particularly clear visualization.

In [ ]:
options = fk.GenerationOptions(max_vertices=3, allow_self_loops=True)
options.add_vertex_allow(["V_3_SCALAR_000"])

generated = model.generate_diagrams(
    incoming=["scalar_0"],
    outgoing=["scalar_0", "scalar_0"],
    loops=1,
    options=options,
)
{
    "diagrams": len(generated),
    "topologies_considered": generated.report.topology_count,
    "interaction_assignments": generated.report.interaction_assignment_count,
}

## Inspect a typed diagram

Vertices and edges retain physics metadata. Methods ending in `_expression` return native `symbolica.core.Expression` objects, ready for symbolic manipulation. Both the diagram and its overall factor have native rich displays, shown in the standalone cells below.

In [ ]:
diagram = next(
    item
    for item in generated.diagrams
    if all(edge.source != edge.target for edge in item.edges)
)
diagram.validate()
factor = diagram.overall_factor_expression()
{
    "name": diagram.name,
    "loops": diagram.loop_count,
    "vertices": len(diagram.vertices),
    "edges": len(diagram.edges),
}

In [ ]:
diagram

In [ ]:
factor.formatted()

In [ ]:
[
    {
        "id": edge.id,
        "from": edge.source,
        "to": edge.target,
        "particle": edge.particle_name,
        "pdg": edge.particle_pdg,
    }
    for edge in diagram.edges
]

## Where to go next

- **01 — Models and diagrams** covers particle/parameter lookup, parameter cards, loop ranges, graph round trips, and loop-momentum bases.
- **02 — CFF and Symbolica** constructs a Cross-Free Family representation and converts it to a Symbolica expression.
- **03 — Kinematics and jets** covers the mostly-minus metric, boosts, rotations, angular distances, and generalized-​\(k_T\) clustering.